# 건강검진 코호트 EDA + 예측모델 6종 비교

데이터: `checkups.parquet` — 10만명 × 5년 분기검진 (180만 행)

**과제**: 과거 검진 이력 → **다음 분기 '위험' 전환 예측**

> ⚠️ 시계열이므로 **사람 단위 분할** 필수 (행 단위로 나누면 같은 사람이 train/test에 걸쳐 누출)

## 0. 환경 준비

In [ ]:
!pip -q install lightgbm xgboost pyarrow
!apt-get install -y fonts-nanum > /dev/null 2>&1

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings, time
import matplotlib.font_manager as fm
fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
plt.rc('font', family='NanumGothic')
plt.rc('axes', unicode_minus=False)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font='NanumGothic')
print('준비 완료')

## 1. 데이터 로드

**코랩 왼쪽 [파일] 탭(폴더 아이콘)에 `checkups.parquet` 드래그해서 업로드** → 업로드 완료(진행 원이 사라짐)까지 기다린 뒤 아래 실행.

> ⚠️ 런타임이 끊기면 업로드한 파일도 사라짐 → 재연결 시 다시 올려야 함 (132MB, 몇 분 소요)

In [ ]:
import os

PATH = '/content/checkups.parquet'   # 파일 탭에 올린 위치

assert os.path.exists(PATH), (
    '파일이 없음. 코랩 왼쪽 [파일] 탭(폴더 아이콘)에 checkups.parquet 을 드래그해서 '
    '업로드가 끝난 뒤 이 셀을 다시 실행하세요.')
print(f'파일 확인 {os.path.getsize(PATH)/1e6:.0f}MB')

df = pd.read_parquet(PATH)
print(f'{len(df):,} 행 · {df.person_id.nunique():,} 명 · {df.quarter.nunique()} 분기')
df.head()

## 2. EDA — 기본 정보 / 결측

In [ ]:
print('=== 형태 ===');  print(df.shape)
print('\n=== 타입 ===');  print(df.dtypes)
print('\n=== 결측 ===');  print(df.isna().sum()[lambda s: s>0] if df.isna().any().any() else '없음')
print('\n=== 인원별 관측 수 ===')
cnt = df.groupby('person_id').size()
print(cnt.describe())
print(f'\n미수검(20회 미만) 비율: {(cnt<20).mean()*100:.1f}%')
df.describe().T.round(2)

## 3. EDA — 인구 분포 (연령·성별·흡연·판정)

In [ ]:
base = df[df.quarter == 0].copy()
def band(a):
    return '20대이하' if a<30 else '30대' if a<40 else '40대' if a<50 else '50대' if a<60 else '60대' if a<70 else '70대+'
base['연령대'] = base.age.map(band)
order = ['20대이하','30대','40대','50대','60대','70대+']

fig, ax = plt.subplots(2, 2, figsize=(14, 9))
sns.countplot(data=base, x='연령대', hue='sex', order=order, ax=ax[0,0]); ax[0,0].set_title('연령대×성별 분포')
sns.countplot(data=df, x='grade', hue='sex', ax=ax[0,1]); ax[0,1].set_title('종합판정×성별 (0정상 1주의 2위험)')
sns.countplot(data=base, x='smoker', hue='sex', ax=ax[1,0]); ax[1,0].set_title('흡연×성별')
sns.histplot(data=base, x='age', hue='sex', bins=30, ax=ax[1,1]); ax[1,1].set_title('연령 분포')
plt.tight_layout(); plt.show()

print('판정 비율:', (df.grade.value_counts(normalize=True).sort_index()*100).round(1).to_dict())
print('흡연율: 전체 %.1f%% / 남 %.1f%% / 여 %.1f%%' % (
    base.smoker.mean()*100, base[base.sex=='M'].smoker.mean()*100, base[base.sex=='F'].smoker.mean()*100))

## 4. EDA — 검진 지표 분포 & 이상치

In [ ]:
METRICS = ['bmi','waist','systolic','diastolic','fbs','total_chol',
           'triglyceride','hdl','ldl','hemoglobin','ast','alt','ggt','creatinine']

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
for ax, c in zip(axes.ravel(), METRICS):
    sns.histplot(df[c], bins=60, ax=ax); ax.set_title(c)
for ax in axes.ravel()[len(METRICS):]: ax.axis('off')
plt.tight_layout(); plt.show()

# IQR 기반 이상치 비율
print('=== 이상치 비율 (IQR 1.5배 기준) ===')
for c in METRICS:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    out = ((df[c] < q1-1.5*iqr) | (df[c] > q3+1.5*iqr)).mean()*100
    print(f'  {c:14} {out:5.2f}%')

## 5. EDA — 상관관계

In [ ]:
corr = df[METRICS + ['age','grade']].corr()
plt.figure(figsize=(12, 9))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, square=True, cbar_kws={'shrink':.8})
plt.title('지표 상관관계'); plt.tight_layout(); plt.show()

print('판정(grade)과 상관 높은 순:')
print(corr['grade'].drop('grade').abs().sort_values(ascending=False).round(3))

## 6. EDA — 시계열 추세 (5년)

In [ ]:
q = df.groupby('quarter').agg(
    bmi=('bmi','mean'), systolic=('systolic','mean'), fbs=('fbs','mean'),
    ldl=('ldl','mean'), 위험비율=('grade', lambda s: (s==2).mean()*100)).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))
for c in ['bmi','systolic','fbs','ldl']:
    ax[0].plot(q.quarter, q[c]/q[c].iloc[0]*100, marker='o', label=c)
ax[0].set_title('지표 평균 추이 (첫 분기=100)'); ax[0].set_xlabel('분기'); ax[0].legend()
ax[1].plot(q.quarter, q.위험비율, marker='o', color='crimson')
ax[1].set_title('위험 판정 비율(%) 추이'); ax[1].set_xlabel('분기')
plt.tight_layout(); plt.show()

# 판정 전이 행렬
d = df.sort_values(['person_id','quarter'])
d['next_grade'] = d.groupby('person_id')['grade'].shift(-1)
trans = pd.crosstab(d.grade, d.next_grade, normalize='index').round(3)
print('판정 전이확률 (행=현재, 열=다음분기)'); print(trans)

## 7. EDA — 개인 궤적 샘플

In [ ]:
ids = df.person_id.drop_duplicates().sample(6, random_state=0).tolist()
fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for ax, pid in zip(axes.ravel(), ids):
    g = df[df.person_id==pid].sort_values('quarter')
    ax.plot(g.quarter, g.systolic, marker='o', label='수축기')
    ax.plot(g.quarter, g.fbs, marker='s', label='공복혈당')
    ax2 = ax.twinx(); ax2.step(g.quarter, g.grade, color='gray', alpha=.5, where='mid')
    ax2.set_ylim(-0.2, 2.2); ax2.set_yticks([0,1,2])
    ax.set_title(f'person {pid} ({g.sex.iloc[0]}, {g.age.iloc[0]:.0f}세)'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. 분석 대상 3만 명 샘플 (사람 단위)

In [ ]:
N_PEOPLE = 30_000
rng = np.random.default_rng(42)
sample_ids = rng.choice(df.person_id.unique(), N_PEOPLE, replace=False)
sdf = df[df.person_id.isin(sample_ids)].copy()
print(f'{len(sdf):,} 행 · {sdf.person_id.nunique():,} 명')

## 9. 특성 공학 — lag / 변화량 / 이동평균

In [ ]:
s = sdf.sort_values(['person_id','quarter']).copy()
g = s.groupby('person_id')

KEY = ['bmi','systolic','diastolic','fbs','ldl','hdl','triglyceride','waist','ggt']
for c in KEY:
    s[f'{c}_lag1'] = g[c].shift(1)
    s[f'{c}_d1']   = s[c] - s[f'{c}_lag1']                    # 직전 대비 변화
    s[f'{c}_ma4']  = g[c].transform(lambda x: x.rolling(4, min_periods=1).mean())
    s[f'{c}_std4'] = g[c].transform(lambda x: x.rolling(4, min_periods=2).std())
s['grade_lag1'] = g['grade'].shift(1)
s['severity_lag1'] = g['severity'].shift(1)
s['is_male'] = (s.sex=='M').astype(int)
s['smoker_i'] = s.smoker.astype(int)

# 타겟: 다음 분기 위험
s['target'] = (g['grade'].shift(-1) == 2).astype('float')
s = s.dropna(subset=['target'])

DROP = ['person_id','quarter','date','sex','smoker','grade','severity','target']
FEATS = [c for c in s.columns if c not in DROP]
print(f'샘플 {len(s):,} · 특성 {len(FEATS)}개 · 양성(다음분기 위험) {s.target.mean()*100:.1f}%')
print(FEATS)

## 10. 사람 단위 분할 (누출 방지)

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

X = s[FEATS]; y = s['target'].values; groups = s['person_id'].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=0)
tr, te = next(gss.split(X, y, groups))
Xtr, Xte, ytr, yte = X.iloc[tr], X.iloc[te], y[tr], y[te]
print(f'train {len(Xtr):,}행 / {s.person_id.iloc[tr].nunique():,}명')
print(f'test  {len(Xte):,}행 / {s.person_id.iloc[te].nunique():,}명')
print('겹치는 사람:', len(set(s.person_id.iloc[tr]) & set(s.person_id.iloc[te])), '(0이어야 정상)')

## 11. 모델 6종 학습·비교

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, roc_curve
import lightgbm as lgb, xgboost as xgb

MODELS = [
    ('LogisticRegression', make_pipeline(SimpleImputer(strategy='median'), StandardScaler(),
                                         LogisticRegression(max_iter=1000))),
    ('DecisionTree',       make_pipeline(SimpleImputer(strategy='median'),
                                         DecisionTreeClassifier(max_depth=8, random_state=0))),
    ('RandomForest',       make_pipeline(SimpleImputer(strategy='median'),
                                         RandomForestClassifier(n_estimators=300, max_depth=14,
                                                                n_jobs=-1, random_state=0))),
    ('HistGradientBoosting', HistGradientBoostingClassifier(max_iter=400, learning_rate=0.05, random_state=0)),
    ('LightGBM',           lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                                              n_jobs=-1, random_state=0, verbose=-1)),
    ('XGBoost',            xgb.XGBClassifier(n_estimators=500, learning_rate=0.05, max_depth=6,
                                             n_jobs=-1, random_state=0, tree_method='hist',
                                             eval_metric='logloss')),
]

rows, curves = [], {}
for name, m in MODELS:
    t0 = time.time(); m.fit(Xtr, ytr); fit_s = time.time()-t0
    p = m.predict_proba(Xte)[:,1]
    pred = (p>=0.5).astype(int)
    rows.append({'모델':name, 'AUC':roc_auc_score(yte,p), 'PR-AUC':average_precision_score(yte,p),
                 'F1':f1_score(yte,pred), 'acc':accuracy_score(yte,pred), '학습(초)':round(fit_s,1)})
    curves[name] = roc_curve(yte, p)
    print(f"{name:22} AUC={rows[-1]['AUC']:.4f}  PR-AUC={rows[-1]['PR-AUC']:.4f}  F1={rows[-1]['F1']:.3f}  {fit_s:.1f}s")

res = pd.DataFrame(rows).sort_values('AUC', ascending=False).reset_index(drop=True)
res.round(4)

## 12. 결과 시각화 (ROC / 성능 비교)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for name, (fpr, tpr, _) in curves.items():
    auc = res.loc[res['모델']==name, 'AUC'].values[0]
    ax[0].plot(fpr, tpr, label=f'{name} ({auc:.3f})')
ax[0].plot([0,1],[0,1],'k--',alpha=.4); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR')
ax[0].set_title('ROC 곡선'); ax[0].legend(fontsize=8)

r = res.melt(id_vars='모델', value_vars=['AUC','PR-AUC','F1'], var_name='지표', value_name='값')
sns.barplot(data=r, x='값', y='모델', hue='지표', ax=ax[1]); ax[1].set_title('성능 비교')
plt.tight_layout(); plt.show()

## 13. 특성 중요도 (최고 성능 모델)

In [ ]:
best_name = res.iloc[0]['모델']
best = dict(MODELS)[best_name]
imp = getattr(best, 'feature_importances_', None)
if imp is None and hasattr(best, 'steps'):
    imp = getattr(best.steps[-1][1], 'feature_importances_', None)

if imp is not None:
    fi = pd.Series(imp, index=FEATS).sort_values(ascending=False).head(20)
    plt.figure(figsize=(8, 6)); sns.barplot(x=fi.values, y=fi.index)
    plt.title(f'{best_name} — 특성 중요도 상위 20'); plt.tight_layout(); plt.show()
else:
    print(f'{best_name}은 feature_importances_ 미제공')

## 14. 학습곡선 — 표본 크기별 성능 (샘플링 타당성 검증)

In [ ]:
sizes = [2_000, 5_000, 10_000, 20_000, 30_000]
curve = []
train_ids = s.person_id.iloc[tr].unique()
for n in sizes:
    n = min(n, len(train_ids))
    ids = rng.choice(train_ids, n, replace=False)
    mask = s.person_id.iloc[tr].isin(ids).values
    m = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=63,
                           n_jobs=-1, random_state=0, verbose=-1)
    m.fit(Xtr[mask], ytr[mask])
    a = roc_auc_score(yte, m.predict_proba(Xte)[:,1])
    curve.append({'인원': n, 'AUC': a}); print(f'  {n:>6,}명  AUC={a:.4f}')

cv = pd.DataFrame(curve)
plt.figure(figsize=(7,4)); plt.plot(cv.인원, cv.AUC, marker='o')
plt.xlabel('학습 인원 수'); plt.ylabel('AUC'); plt.title('학습곡선 (LightGBM)')
plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 15. 요약

- 사람 단위 분할로 누출 차단 확인
- 6모델 동일 조건 비교 (AUC / PR-AUC / F1 / 학습시간)
- 특성 중요도로 어떤 지표가 위험 전환을 예측하는지 확인
- 학습곡선으로 표본 크기 타당성 검증

다음 단계: 최고 모델 저장 → `joblib.dump(best, 'risk_model.pkl')` → API에서 로드해 서빙

In [ ]:
import joblib
joblib.dump({'model': best, 'features': FEATS}, 'risk_model.pkl')
from google.colab import files
files.download('risk_model.pkl')   # 로컬로 내려받아 API에 연결
print('저장 완료:', best_name)